In [ ]:
import pandas as pd
df = pd.read_excel(r'C:\Users\josperez\Downloads\Early Churn Risk_Data.xlsx',engine='calamine')
display(df.info())
display(df.describe(include='all'))
display(df.columns)
display(df.dtypes)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4506 entries, 0 to 4505
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID                 4506 non-null   object        
 1   N° Abonado         4506 non-null   object        
 2   Documento          4506 non-null   int64         
 3   Cliente            4506 non-null   object        
 4   Fecha Contrato     4506 non-null   datetime64[ns]
 5   Estatus            4506 non-null   object        
 6   Teléfono           4505 non-null   object        
 7   Teléfono Casa      267 non-null    object        
 8   Teléfono Adic      3342 non-null   object        
 9   Nombre Franquicia  4506 non-null   object        
 10  Ciudad             4506 non-null   object        
 11  Vendedor           4506 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 422.6+ KB


None

,ID,N° Abonado,Documento,Cliente,Fecha Contrato,Estatus,Teléfono,Teléfono Casa,Teléfono Adic,Nombre Franquicia,Ciudad,Vendedor
count,4506,4506,4.506000e+03,4506,4506,4506,4.505000e+03,2.670000e+02,3.342000e+03,4506,4506,4506
unique,4506,4506,NaN,4437,NaN,7,4.403000e+03,2.610000e+02,3.283000e+03,43,101,687
top,'CONT9096B3F16FD44818',A76746,NaN,ROGELIN ZENAIDA ZAMORA FREITEZ,NaN,CORTADO,4.127484e+09,2.418720e+09,4.261448e+09,FIBEX VALENCIA,VALENCIA,NELSON GARCIA AGENTE AUTORIZADO GLOBAL TELECOM
freq,1,1,NaN,4,NaN,2744,4.000000e+00,2.000000e+00,4.000000e+00,884,804,90
mean,NaN,NaN,2.839978e+13,NaN,2026-02-06 01:31:23.888149248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000070e+05,NaN,2026-01-02 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,1.409768e+07,NaN,2026-01-19 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,1.973257e+07,NaN,2026-02-04 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,2.691710e+07,NaN,2026-02-25 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,1.279692e+17,NaN,2026-03-26 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Index(['ID', 'N° Abonado', 'Documento', 'Cliente', 'Fecha Contrato', 'Estatus',
       'Teléfono', 'Teléfono Casa', 'Teléfono Adic', 'Nombre Franquicia',
       'Ciudad', 'Vendedor'],
      dtype='object')

ID                           object
N° Abonado                   object
Documento                     int64
Cliente                      object
Fecha Contrato       datetime64[ns]
Estatus                      object
Teléfono                     object
Teléfono Casa                object
Teléfono Adic                object
Nombre Franquicia            object
Ciudad                       object
Vendedor                     object
dtype: object

In [ ]:
import pandas as pd
import numpy as np

# =========================================================================
# 1. CARGA DE DATOS Y PREPARACIÓN DE FECHAS
# =========================================================================
print("⏳ Cargando datos y formateando fechas...")
ruta = r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\silver_data\Tickets_Silver_Master.parquet'
df = pd.read_parquet(ruta)

# Convertimos a datetime las columnas con hora exacta para cálculos precisos
df['Fecha Apertura'] = pd.to_datetime(df['Fecha Apertura'])
df['Fecha Cierre'] = pd.to_datetime(df['Fecha Cierre'])
df['Fecha Impresion'] = pd.to_datetime(df['Fecha Impresion'])

# =========================================================================
# 2. ANÁLISIS 1: RECURRENCIA POR LA MISMA CAUSA (FCR PURO)
# =========================================================================
print("🔍 Calculando Delta de Tiempo y Supervivencia de Soluciones...")
# Ordenamos cronológicamente por contrato y por falla
df = df.sort_values(['N° Contrato', 'Detalle Orden', 'Fecha Apertura'])

# Traemos la fecha de apertura del SIGUIENTE ticket (mismo cliente, misma falla)
df['Siguiente_Apertura'] = df.groupby(['N° Contrato', 'Detalle Orden'])['Fecha Apertura'].shift(-1)

# Supervivencia = Fecha en que volvió a fallar - Fecha en que se "resolvió" la vez anterior
df['Supervivencia_Dias'] = (df['Siguiente_Apertura'] - df['Fecha Cierre']).dt.total_seconds() / 86400.0

# Evitamos que tickets solapados generen días negativos y dañen el promedio
df['Supervivencia_Dias'] = df['Supervivencia_Dias'].clip(lower=0)

# Agrupación base (El truco de las comas para ver el historial en una celda)
df_causas = df.groupby(['Quincena Evaluada', 'N° Contrato', 'Detalle Orden']).agg(
    Repeticiones = ('N° Contrato', 'size'),
    Fechas_Atendidas = ('Fecha Apertura Date', lambda x: '; '.join(x.dropna().astype(str).unique())),
    Soluciones = ('Solucion Aplicada', lambda x: '; '.join(x.dropna().astype(str).unique())),
    Promedio_Supervivencia_Dias = ('Supervivencia_Dias', 'mean')
).reset_index()

df_causas['Promedio_Supervivencia_Dias'] = df_causas['Promedio_Supervivencia_Dias'].round(1)

# Aislamos los casos críticos de MISMA CAUSA (Los ~8,000 históricos)
df_recurrentes_detalle = df_causas[df_causas['Repeticiones'] > 1].copy()

# =========================================================================
# 3. EL PARETO ESTRATÉGICO (80/20) PARA LA MISMA CAUSA
# =========================================================================
print("📊 Generando Análisis de Pareto...")
ranking_fallas = df_recurrentes_detalle.groupby('Detalle Orden').agg(
    Casos_Afectados = ('N° Contrato', 'nunique'),
    Total_Tickets_Involucrados = ('Repeticiones', 'sum'), # Nombre claro
    Supervivencia_Promedio_Dias = ('Promedio_Supervivencia_Dias', 'mean')
).sort_values('Total_Tickets_Involucrados', ascending=False)

# Matemáticas del Pareto
total_reaperturas = ranking_fallas['Total_Tickets_Involucrados'].sum()
ranking_fallas['%_Del_Total'] = (ranking_fallas['Total_Tickets_Involucrados'] / total_reaperturas) * 100
ranking_fallas['%_Acumulado_Pareto'] = ranking_fallas['%_Del_Total'].cumsum()

# Limpieza visual de decimales
ranking_fallas['Supervivencia_Promedio_Dias'] = ranking_fallas['Supervivencia_Promedio_Dias'].round(1)
ranking_fallas['%_Del_Total'] = ranking_fallas['%_Del_Total'].round(2)
ranking_fallas['%_Acumulado_Pareto'] = ranking_fallas['%_Acumulado_Pareto'].round(2)

# Resumen Temporal (Evolución por Quincena para la misma causa)
ranking_quincena = df_recurrentes_detalle.groupby('Quincena Evaluada').agg(
    Casos_Afectados = ('N° Contrato', 'nunique'),
    Total_Tickets_Involucrados = ('Repeticiones', 'sum'),
    Supervivencia_Promedio_Dias = ('Promedio_Supervivencia_Dias', 'mean')
).sort_values('Quincena Evaluada', ascending=True)

ranking_quincena['Supervivencia_Promedio_Dias'] = ranking_quincena['Supervivencia_Promedio_Dias'].round(1)

# =========================================================================
# 4. ANÁLISIS 2: FRICCIÓN GENERAL (EL UNIVERSO DE LOS 38K)
# =========================================================================
print("🔄 Analizando el Universo de Fricción General (Todos los reclamos múltiples)...")
# Ordenamos solo por contrato y fecha para ver el viaje real del cliente
df_ordenado_cliente = df.sort_values(['N° Contrato', 'Fecha Apertura'])

# Agrupamos por Quincena y Contrato para sacar el Total Real por cliente
df_general = df_ordenado_cliente.groupby(['Quincena Evaluada', 'N° Contrato']).agg(
    Total_Tickets = ('N° Contrato', 'size'),
    
    # Unimos todas las causas con una flecha, sean iguales o diferentes
    Viaje_Fallas = ('Detalle Orden', lambda x: ' ➔ '.join(x.dropna().astype(str))), 
    Cantidad_Causas_Distintas = ('Detalle Orden', 'nunique')
).reset_index()

# EL UNIVERSO: Absolutamente todos los clientes con > 1 ticket (Tus 38,000)
df_friccion_total = df_general[df_general['Total_Tickets'] > 1].copy()

# Ranking general de combinaciones (te mostrará tanto "Lentitud ➔ Lentitud" como "Lentitud ➔ Corte")
ranking_friccion = df_friccion_total.groupby('Viaje_Fallas').agg(
    Casos_Afectados = ('N° Contrato', 'nunique'),
    Total_Tickets_Involucrados = ('Total_Tickets', 'sum')
).sort_values('Casos_Afectados', ascending=False)


# =========================================================================
# 5. ANÁLISIS 3: REINCIDENCIA POR GRUPO DE TRABAJO (AUDITORÍA FCR)
# =========================================================================
print("👥 Calculando Tasa de Reincidencia por Grupo de Trabajo...")
# Si un ticket tiene 'Siguiente_Apertura', significa que falló el FCR
df['FCR_Fallido'] = df['Siguiente_Apertura'].notna()

resumen_grupos = df.groupby('Grupo Trabajo').agg(
    Total_Tickets_Atendidos = ('N° Orden', 'count'),
    Tickets_Reincidentes = ('FCR_Fallido', 'sum'),
    Dias_Prom_Reincidencia = ('Supervivencia_Dias', 'mean')
).reset_index()

# Calculamos la Tasa de Reincidencia %
resumen_grupos['Tasa_Reincidencia_%'] = (resumen_grupos['Tickets_Reincidentes'] / resumen_grupos['Total_Tickets_Atendidos']) * 100

# Limpieza y filtrado
resumen_grupos['Tasa_Reincidencia_%'] = resumen_grupos['Tasa_Reincidencia_%'].round(2)
resumen_grupos['Dias_Prom_Reincidencia'] = resumen_grupos['Dias_Prom_Reincidencia'].round(2)

# Excluimos grupos con muy pocos tickets para no sesgar los porcentajes (opcional, ajustado a 20)
resumen_grupos = resumen_grupos[resumen_grupos['Total_Tickets_Atendidos'] >= 20]

# Ordenamos por los que más generan retrabajo
resumen_grupos = resumen_grupos.sort_values(by='Tickets_Reincidentes', ascending=False)


# =========================================================================
# 6. SELECCIÓN DE COLUMNAS PARA AUDITORÍA (Detalle Total Liviano)
# =========================================================================
columnas_esenciales = [
    'Quincena Evaluada', 'N° Contrato', 'N° Orden', 'Detalle Orden', 
    'Fecha Apertura', 'Fecha Impresion', 'Fecha Cierre', 
    'Solucion Aplicada', 'Duracion_Horas', 'SLA Resolucion Min', 
    'SLA Impresion Min', 'Cumplio_SLA', 'Franquicia', 'Grupo Trabajo'
]
df_reducido = df[columnas_esenciales].copy()

# =========================================================================
# 6.5 NUEVO: LÍNEA DE TIEMPO CRONOLÓGICA POR ABONADO (El Expediente)
# =========================================================================
print("📅 Generando historial cronológico detallado por abonado...")

# Identificamos a los abonados que sufrieron fricción (los que tienen > 1 ticket)
contratos_con_friccion = df_friccion_total['N° Contrato'].unique()

# Filtramos la base de datos original solo para estos abonados críticos
df_historial_cronologico = df[df['N° Contrato'].isin(contratos_con_friccion)].copy()

# Ordenamos jerárquicamente por Contrato y luego por Fecha
df_historial_cronologico = df_historial_cronologico.sort_values(
    by=['N° Contrato', 'Fecha Apertura'], 
    ascending=[True, True]
)

# Cruzamos la tabla para traernos el "Viaje_Fallas" desde df_friccion_total
df_historial_cronologico = df_historial_cronologico.merge(
    df_friccion_total[['N° Contrato', 'Viaje_Fallas']], 
    on='N° Contrato', 
    how='left'
)

# Filtramos las columnas que importan para la auditoría + la nueva visualización del viaje
columnas_expediente = columnas_esenciales + ['Viaje_Fallas']
df_historial_cronologico = df_historial_cronologico[columnas_expediente]

# =========================================================================
# 7. EXPORTACIÓN A EXCEL (Multi-hojas)
# =========================================================================
print("💾 Guardando el reporte estratégico en Excel...")
nombre_archivo = r'C:\Users\josperez\Downloads\Analisis_Recurrencia_FCR_Fibex2.xlsx'

with pd.ExcelWriter(nombre_archivo, engine='xlsxwriter') as writer:
    # 1. El Pareto del Subconjunto Crítico (Misma Causa - Los 8k)
    ranking_fallas.to_excel(writer, sheet_name='Pareto_Misma_Causa', index=True)
    
    # 2. El Ranking del Universo Total (Los 38k)
    ranking_friccion.to_excel(writer, sheet_name='Ranking_Friccion_General', index=True)
    
    # 3. Evolución Temporal (Misma Causa)
    ranking_quincena.to_excel(writer, sheet_name='Resumen_por_Quincena', index=True)
    
    # 4. Auditoría por Grupo de Trabajo
    resumen_grupos.to_excel(writer, sheet_name='Auditoria_Grupo_Trabajo', index=False)
    
    # 5. Detalle de los contratos críticos (Misma Causa - Los 8k)
    df_recurrentes_detalle.to_excel(writer, sheet_name='Detalle_Contratos_Misma', index=False)
    
    # 6. Detalle del Universo de contratos múltiples (Los 38k)
    df_friccion_total.to_excel(writer, sheet_name='Detalle_Contratos_Todos', index=False)
    
    # NUEVO: 7. El "Expediente" cronológico exacto para auditoría visual
    df_historial_cronologico.to_excel(writer, sheet_name='Historial_Cronologico', index=False)
    
    # 8. Data origen optimizada
    df_reducido.to_excel(writer, sheet_name='Detalle_Total', index=False)

print(f"✅ ¡Éxito! Reporte corporativo generado en: {nombre_archivo}")

⏳ Cargando datos y formateando fechas...
🔍 Calculando Delta de Tiempo y Supervivencia de Soluciones...
📊 Generando Análisis de Pareto...
🔄 Analizando el Universo de Fricción General (Todos los reclamos múltiples)...
👥 Calculando Tasa de Reincidencia por Grupo de Trabajo...
📅 Generando historial cronológico detallado por abonado...
💾 Guardando el reporte estratégico en Excel...
✅ ¡Éxito! Reporte corporativo generado en: C:\Users\josperez\Downloads\Analisis_Recurrencia_FCR_Fibex2.xlsx


Creacion de consulta para la identificacion de los clientes tomados para la muestra de llamadas:
    
    Aquellos que entre dentro del periodo escogido de generacion de contrato
    Agrupacion por marca de clase de dias de morosidad o de vencimiento de pago. (30-60 dias)

In [63]:
import duckdb
import polars as pl

ruta_early_churn_risk = r"C:\Users\josperez\Downloads\Early_Churn_Risk_Data.xlsx"
ruta_ecr = ruta_early_churn_risk 

# Leemos con calamine (excelente elección para rendimiento)
ecr = pl.read_excel(ruta_ecr, engine='calamine')

# Usamos un CTE (WITH) para calcular "dias" primero y luego poder usarlo en el CASE
query = """--sql
    WITH base_calculada AS (
        SELECT 
            *,
            -- Calculamos la diferencia de días. 
            -- Asegúrate que ambas columnas sean tipo Date/Timestamp
            ("Último Corte Finalizado" - "Fecha Últ. Pago") AS dias
        FROM ecr
    )
    SELECT 
        *,
        -- Ahora sí podemos usar "dias" directamente
        CASE 
            WHEN dias >= 30 AND dias < 60 THEN 30
            WHEN dias >= 60 AND dias < 120 THEN 60
            WHEN dias >= 120 THEN 120
            ELSE 0 
        END AS marca_clase
    FROM base_calculada
    WHERE estatus != 'ACTIVO'
    ORDER BY dias DESC;
"""

# Ejecutamos y mostramos
df = duckdb.sql(query).to_df()
display(df)

,ID,N° Abonado,Documento,Cliente,Fecha Contrato,Estatus,Suscripción,Fecha Últ. Pago,Teléfono,Teléfono Adic,Nombre Franquicia,Ciudad,Último Corte Finalizado,Serv/Paquete,dias,marca_clase
0,'CONT618A550738205907',BQ38844,29805386,DANIELA ANAHIS CHIRINOS BRICEñO,2026-01-09,CORTADO,23.20,2026-01-10,4245677484,4125233966,FIBEX BARQUISIMETO,BARQUISIMETO,2026-03-12,FTTH_400,61,60
1,'CONT5BCF768C8C974572',A74039,6317368,PASCUAL RAMON ARVELO MEDINA,2026-01-05,CORTADO,23.20,2026-01-05,4243285030,4243285030,FIBEX ARAGUA,TURMERO,2026-03-05,FTTH_400,59,30
2,'CONT5C1F05E50F670270',A74110,20356208,JUAN JOSE IBARRA PARRA,2026-01-05,CORTADO,30.00,2026-01-05,4141493873,<NA>,FIBEX ARAGUA,MARACAY,2026-03-05,FTTH_300,59,30
3,'CONT5C10FC3C0C720769',A74099,25367034,YAMILEXYS DEL CARMEN MERO BRIZUELA,2026-01-05,CORTADO,23.20,2026-01-05,4247746875,4247746875,FIBEX ARAGUA,LA VICTORIA,2026-03-05,FTTH_200,59,30
4,'CONT5D687AA52DB60720',A74187,28302254,MAYRA ISABEL MACIAS ARREAGA,2026-01-06,CORTADO,23.20,2026-01-06,4241960560,4128940954,FIBEX ARAGUA,EL LIMóN,2026-03-05,FTTH_400,58,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4271,'CONTA708110C43667154',LG21223,3311269,MARITZA RAMOS GONZALEZ,2026-03-03,REEMBOLSO PAGADOS,25.00,2026-03-03,4125576639,2123328278,FIBEX LA GUAIRA,LA GUAIRA,NaT,None,<NA>,0
4272,'CONTA709D6574CE54089',AP6984,21147677,JUNIOR USECHE,2026-03-03,REEMBOLSO PAGADOS,46.40,2026-03-03,4261406559,4243381789,FIBEX SAN FERNANDO DE APURE,SAN FERNANDO DE APURE,NaT,None,<NA>,0
4273,'CONTA71476BBC7F79729',V137845,18628671,AMALIA MERCEDES SILVA ROJAS,2026-03-03,REEMBOLSO PAGADOS,29.00,2026-03-03,4244574949,4244045193,FIBEX VALENCIA,VALENCIA,NaT,None,<NA>,0
4274,'CONTA1E2240CF5F06495',LCH85541,26421904,FREDDY ALEXANDER PAISAN CARRERA,2026-02-27,REEMBOLSO PAGADOS,23.20,2026-02-27,4248615330,4248279937,FIBEX ANZOATEGUI,CUMANA,NaT,None,<NA>,0
